# MultiTree

The `toytree.MultiTree` class object is used to represent a collection of `ToyTree` objects and includes attributes and methods for describing this set or performing operations on it. Common examples of tree sets include bootstrap replicate samples or posterior distributions of sampled trees; common operations on sets of trees include *consensus tree inference*, computing discordance or distance statistics, and plotting tree grids or cloud trees.

In [92]:
import toytree

## Generating MultiTrees

MultiTree objects can be generated from a list of Toytrees or newick strings, or by parsing a file, url, or string of text that includes newick trees separated by newlines. The convenience function `toytree.mtree()` can be used to parse multitree input data similar to how the function `toytree.tree` is used to parse individual trees, and supports the same file formats.

### From tree data
Below is an example multi-newick string representing multiple trees as newick strings separated by newlines. You can create a MultiTree from this input data, entered as a string or filepath, by passing it to the `toytree.mtree()` convenience parsing function. Each tree will be parsed individually and stored as a list of `ToyTree` objects contained within a returned `MultiTree` object.

In [93]:
multinewick = """\
(((a:1,b:1):1,(d:1.5,e:1.5):0.5):1,c:3);
(((a:1,d:1):1,(b:1,e:1):1):1,c:3);
(((a:1.5,b:1.5):1,(d:1,e:1):1.5):1,c:3.5);
(((a:1.25,b:1.25):0.75,(d:1,e:1):1):1,c:3);
(((a:1,b:1):1,(d:1.5,e:1.5):0.5):1,c:3);
(((b:1,a:1):1,(d:1.5,e:1.5):0.5):2,c:4);
(((a:1.5,b:1.5):0.5,(d:1,e:1):1):1,c:3);
(((b:1.5,d:1.5):0.5,(a:1,e:1):1):1,c:3);
"""

In [94]:
# create an mtree from a string, list of strings, url, or file.
mtree1 = toytree.mtree(multinewick)
mtree1

<toytree.MultiTree ntrees=8>

### From a collection of trees
Similarly, you can create a `MultiTree` by providing a collection of `ToyTree` objects to the `toytree.mtree` function. Here we generate a list of 50 random coalescent trees and pass the list as input to create a new `MultiTree`.

In [95]:
# generate 50 random coalescent trees each with 6 tips
coaltrees = [toytree.rtree.coaltree(k=6) for i in range(50)]

In [96]:
# create a MultiTree from a list of ToyTrees
mtree2 = toytree.mtree(coaltrees)
mtree2

<toytree.MultiTree ntrees=50>

## Indexable and Iterable
One or more trees can be indexed or sliced from a `MultiTree`, and sequential trees can be accessed through iteration. The trees themselves are stored in the `.treelist` attribute of the `MultiTree` object as a list. This can be modified to remove, add, or reorder the trees. Several example operations are shown below for accessing one or more trees.

In [97]:
# get first tree
mtree1[0]

In [98]:
# get all trees
mtree1[:]

In [99]:
# slice the first three trees
mtree1[:3]

In [100]:
# iterate over ToyTrees in a MultiTree
for tree in mtree1:
    print(tree)

In [101]:
# re-arrange trees in the treelist to send the first to be last
mtree1.treelist = mtree1.treelist[1:] + [mtree1.treelist[0]]
mtree1[:]

## Attributes and types of tree sets
Most of the time `MultiTree` objects are used to hold a collection of trees that all share the same tip labels, such as a collection of bootstrap replicates. But, in other cases, a `MultiTree` could hold a collection of unrelated trees, in which case some of the built-in functions for comparing trees (such as consensus tree inference) will raise an error, but it still provides a useful container for drawing trees. These methods will raise a ToyTreeError when attempted if the tree set is a mixed collection of trees. The  `MultiTree` class contains several functions to quickly check attributes of the tree set to examine the number of trees, whether they share the same tip names, and whether the trees are rooted or ultrametric.

In [102]:
mtree1.ntrees

8

In [103]:
mtree1.all_tree_tip_labels_same()

True

In [104]:
mtree1.all_tree_topologies_same()

False

In [105]:
mtree1.all_tree_tips_aligned()

False

## Consensus trees
For a full parameter and output-field reference see: [Inference: Consensus Trees](/toytree/infer-consensus/).

A majority-rule consensus tree summarizes the most common non-conflicting splits among a set of input trees. A consensus tree can be inferred from `MultiTree.get_consensus_tree(min_freq=...)`. The returned `ToyTree` is unrooted and stores split support scores in the "support" feature, and edge distance summaries (e.g., `dist_mean`, `dist_std`). There are additional options to summarize other feature data from the set of input trees onto the consensus tree.

In [107]:
# get a consensus tree
ctree = mtree1.get_consensus_tree()

# plot the unrooted tree showing 'support' values
c, a, m = ctree.draw(layout="unr", height=350)
ctree.annotate.add_edge_labels(a, "support", color="grey");

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="314.0px" height="350.0px" viewBox="0 0 314.0 350.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t5eab94700ddd4129857bc51481335d55"> a b d e c 0.75 0.75

## Unique trees
Given a set of trees it is useful to be able to pull out just the unique topologies from the set. The function `get_unique_topologies()` returns a list of `(tree, int)` tuples from a `MultiTree` with each unique topology paired with its number of occurrences in the set. Note, this condenses all trees with the same topology into a single representative, using the first occurrence as the returned tree, thus branch length variation is not retained.

In [108]:
# get (tree, count) for each unique topology in the MultiTree
mtree1.get_unique_topologies()

[(<toytree.ToyTree at 0x7a751f127ec0>, 6),
 (<toytree.ToyTree at 0x7a751d9f2ea0>, 1),
 (<toytree.ToyTree at 0x7a751ef42db0>, 1)]

## Drawing with MultiTrees

### Grid tree drawings
See [MultiTree.draw()](drawing-multitree-grids) for a detailed description of MultiTree grid drawings.

The `MultiTree.draw()` method returns a drawing with multiple trees displayed on a grid. The `shape` and `idxs` arguments can be used to designate the grid layout and select which trees to show. All standard tree drawing style arguments are accepted. The `fixed_order` argument is often useful in this context to fix the order of tips to emphasize discordance among trees in a set.

In [111]:
# draw a 2x4 grid of trees 8 trees from a collection
mtree1.draw(
    ts="o", shape=(2, 4), width=600, height=300, fixed_order=["c", "b", "e", "a", "d"]
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="600.0px" height="300.0px" viewBox="0 0 600.0 300.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tf53d86b50500424c8c18c381c0b5b298"> a d b e c a b d e c a b d e c a b d e c b a d e c a b d e c b d a e c a b d e c

### Cloud tree drawings
See [MultiTree.draw_cloud_tree()](drawing-multitree-cloud-trees) for a detailed description of MultiTree cloud tree drawings.

It is sometimes even more informative to plot a number of trees on top of each other to visualize their discordance. These are sometimes called “densitree” plots, or here, “cloud tree plots”.

In [112]:
# draw a cloud tree
mtree1.draw_cloud_tree(
    scale_bar=True,
    edge_style={
        "stroke-opacity": 0.1,
        "stroke-width": 3,
    },
);

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="275.0px" viewBox="0 0 300.0 275.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="tbe50d67ea03c4cff96ec96637b6b0fc2"> 4 3 2 1 0 a d b e c